# SI — TetraSpeck bead precision: the colour counting model, and the three excitation lasers

**Makes up:** two SI figures for SI Note *Calibrating Precision* (Fig. 2d):

1. **Counting model** (`BeadPrecision_CountingModel.svg`) — why the colour precision of all three pixel
   types and all three lasers falls on one curve against $N_X/g_X$: **a)** Monte Carlo check of the
   formula below; **b–d)** bead colour precision against $N_X$, $N_X/(1-A_X)$ and $N_X/g_X$.
2. **By excitation laser** (`BeadPrecision_ByExcitation.svg`) — σ_xy against $N$ and each pixel's
   σ_colour against $N_X/g_X$, binned per laser (488, 561, 638 nm), where the lasers' signal ranges
   overlap. If the lasers agree there, pooling them in Fig. 2d is justified (Reviewer 4, minor 7).

**Input:** `bead_precision.csv`, one row per bead, written by §3 of `../Figure_2d_TetraspeckPrecision.ipynb`
(run that first, with the same `OUT_ROOT` and `FAST_MODE`). The same cuts as Fig. 2d are applied
(`_beads.clean_loc`, `_beads.clean_colour`).

## The counting model

The fit returns, for each pixel type Y, the amplitude $A_Y$ that type would collect if it covered the
whole sensor. It is measured from the $N_Y = f_Y A_Y$ photoelectrons pixel type Y detects ($f_Y$: its share
of the sensor, $\tfrac14, \tfrac12, \tfrac14$ for the Bayer mosaic), so counting (Poisson) noise gives
$\mathrm{var}(A_Y) = A_Y/f_Y$. The colour is the share $a_X = A_X/\sum_Y A_Y$ (relative pixel QE).
Propagating (delta method, no background, independent pixel types):

$$\left(\frac{\sigma_{a_X}}{a_X}\right)^2 = \frac{g_X}{N_X},\qquad
g_X = (1-a_X)^2 + a_X f_X \sum_{Y\neq X}\frac{a_Y}{f_Y}.$$

With equal pixel fractions $g_X = 1-a_X$, the familiar binomial result. On the Bayer mosaic pixel 2
covers twice the area of pixels 1 and 3, and $g_2 > 1-a_2$ (1.7× for the 488 nm beads). Plotting
σ_colour against the effective count $N_X/g_X$ puts every pixel type on $1/\sqrt{N_X/g_X}$ for pure
counting noise. Background, read noise and the fit add a roughly constant factor on top, which the
$a/\sqrt{N}+c$ fit of Fig. 2d absorbs.

Changed from the published Fig. 2d: σ_colour was plotted against the signal on each pixel type with
pixel 2 multiplied by $\sqrt2$; from 2026-11-09 to 2026-11-16 against $N_X/(1-a_X)$.

---
## 0 — Imports and configuration

In [ ]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
from matplotlib.lines import Line2D

# _beads.py lives one folder up, beside Figure_2d_TetraspeckPrecision.ipynb.
sys.path.insert(0, str(Path.cwd().resolve().parent))
import _beads as bd

from pyS3M import PlottingBase
plotter = PlottingBase.PublicationPlotter(dark_background=False)

FAST_MODE = False   # as in Figure_2d_TetraspeckPrecision: which of its runs to read
OUT_ROOT  = Path("/media/u2473660/JSBLeeLabBackup/Data/20261026_Tetraspeck")   # Figure_2d's OUT_ROOT
BEAD_TABLE = OUT_ROOT / ("fast" if FAST_MODE else "full") / "bead_precision.csv"
FIG_DIR = OUT_ROOT / ("fast" if FAST_MODE else "full") / "SI_BeadPrecision"
FIG_DIR.mkdir(parents=True, exist_ok=True)
N_BOOT = 1000
MIN_PER_BIN = 3 if FAST_MODE else 5
EDGES = np.geomspace(10, bd.MAX_PE, 16)

PIXEL_MARKERS = {"B": "o", "G": "s", "R": "^"}
PIXEL_NAMES = {"B": "pixel 1", "G": "pixel 2", "R": "pixel 3"}


def grid(ax):
    ax.grid(color="gray", lw=0.5, ls="--", alpha=0.5)
    ax.set_axisbelow(True)


def log_axes(ax, x=True, y=True):
    for on, axis in ((x, ax.xaxis), (y, ax.yaxis)):
        if on:
            axis.set_major_locator(mticker.LogLocator(base=10, numticks=12))
            axis.set_major_formatter(mticker.FuncFormatter(lambda v, _: f"{v:g}"))
            axis.set_minor_formatter(mticker.NullFormatter())


print(f"Bead table: {BEAD_TABLE}  ({'found' if BEAD_TABLE.exists() else 'NOT FOUND: run Figure_2d §3 first'})")

---
## 1 — Beads (same cuts as Fig. 2d)

In [ ]:
beads = pd.read_csv(BEAD_TABLE, dtype={"laser": str, "power": str})
loc = bd.clean_loc(beads)
colour = bd.clean_colour(bd.colour_long(beads))
colour = colour.assign(n_1ma=colour["pe_px"] / (1 - colour["A_px"]))
print(f"{len(beads):,} beads; {len(loc):,} in the sigma_xy plot; {len(colour):,} bead-pixel pairs in the colour plot")

# expected shares of each laser's beads, from the dye spectra (as SI Table photons_to_pe)
SHARES = {}
for laser in bd.LASERS:
    _, q = bd.bead_effective_qe(laser)
    SHARES[laser] = {ch: q[ch] / sum(q.values()) for ch in bd.CHANNELS}
g_expected = pd.DataFrame({f"{l} nm": bd.colour_count_factor(SHARES[l]) for l in bd.LASERS}).T
one_minus = pd.DataFrame({f"{l} nm": {ch: 1 - SHARES[l][ch] for ch in bd.CHANNELS} for l in bd.LASERS}).T
print("Expected shares a_X:")
print(pd.DataFrame(SHARES).T.round(3))
print("\ng_X / (1 - a_X) at the expected shares:")
print((g_expected / one_minus).round(2))
print("\nMeasured median shares per laser:")
print(beads.groupby("laser")[[f"A_{c}" for c in bd.CHANNELS]].median().round(3))

---
## 2 — Figure: the counting model

**a)** Monte Carlo: for each laser's expected shares and $N$ = 100–10,000 detected photoelectrons,
20,000 frames of Poisson photoelectrons on each pixel type (no background), amplitudes $N_Y/f_Y$, and
the std / mean of the shares. Filled: against $\sqrt{g_X/N_X}$; open: against $\sqrt{(1-a_X)/N_X}$.
On the line = the formula is exact.

**b–d)** Bead colour precision (median per bin of each laser × pixel type, 95 % bootstrap interval)
against $N_X$, $N_X/(1-a_X)$ and $N_X/g_X$. Dashed: pure counting noise, $100/\sqrt{x}$ %. The
printed log-residual SD of $\sigma_\mathrm{colour}\sqrt{x}$ over all bead-pixel pairs measures how well each
axis collapses the nine groups.

In [ ]:
mc = pd.concat([bd.simulate_colour_precision(SHARES[l], n, seed=k).assign(laser=l, n_pe=n)
                for l in bd.LASERS for k, n in enumerate(np.geomspace(100, 1e4, 5))], ignore_index=True)
print("Monte Carlo / formula, range over lasers, pixels and N:")
print(f"  exact g_X : {np.min(mc['simulated'] / mc['exact']):.3f} - {np.max(mc['simulated'] / mc['exact']):.3f}")
print(f"  1 - a_X   : {np.min(mc['simulated'] / mc['one_minus_a']):.3f} - {np.max(mc['simulated'] / mc['one_minus_a']):.3f}")

axes_x = [("pe_px", r"N$_X$ (photoelectrons on pixel X)"),
          ("n_1ma", r"N$_X$/(1$-$a$_X$)"),
          ("n_eff", r"N$_X$/g$_X$")]
spread = {}
for key, _ in axes_x:
    c = colour[colour[key] > 0]
    r = np.log(c["prec_pct"] / 100 * np.sqrt(c[key]))
    groups = np.exp(c.assign(r=r).groupby(["laser", "channel"])["r"].median())
    spread[key] = {"log-residual SD": r.std(), "group median min": groups.min(), "group median max": groups.max(),
                   "max / min": groups.max() / groups.min()}
print("\nsigma_colour x sqrt(x), over bead-pixel pairs (group = laser x pixel type):")
print(pd.DataFrame(spread).T.rename(index=dict((k, l) for k, l in axes_x)).round(3))

fig, axs = plotter.two_column_plot(nrows=1, ncols=4, height=2.2)
ax = axs[0]
for laser in bd.LASERS:
    for ch in bd.CHANNELS:
        m = mc[(mc["laser"] == laser) & (mc["channel"] == ch)]
        ax.scatter(100 * m["exact"], 100 * m["simulated"], marker=PIXEL_MARKERS[ch], s=12,
                   color=bd.LASER_COLOURS[laser], lw=0.6)
        ax.scatter(100 * m["one_minus_a"], 100 * m["simulated"], marker=PIXEL_MARKERS[ch], s=12,
                   facecolor="none", edgecolor=bd.LASER_COLOURS[laser], lw=0.6)
lim = [0.5, 30]
ax.plot(lim, lim, color="k", lw=0.8)
ax.set_xscale("log"); ax.set_yscale("log"); ax.set_xlim(lim); ax.set_ylim(lim)
log_axes(ax)
ax.set_xlabel(r"predicted $\sigma_{colour}$/%")
ax.set_ylabel(r"simulated $\sigma_{colour}$/%")
grid(ax)

for ax, (key, xlabel) in zip(axs[1:], axes_x):
    for laser in bd.LASERS:
        for ch in bd.CHANNELS:
            e = colour[(colour["laser"] == laser) & (colour["channel"] == ch)]
            b = bd.binned_medians(e[key], e["prec_pct"], EDGES, MIN_PER_BIN, N_BOOT).dropna()
            ax.errorbar(b["pe"], b["median"], yerr=[b["median"] - b["lo"], b["hi"] - b["median"]],
                        color=bd.LASER_COLOURS[laser], marker=PIXEL_MARKERS[ch], ms=3, lw=0.8, capsize=1.5)
    x = np.geomspace(EDGES[0], EDGES[-1], 50)
    ax.plot(x, 100 / np.sqrt(x), color="k", lw=0.8, ls="--")
    ax.set_xscale("log"); ax.set_yscale("log")
    ax.set_xlim(EDGES[0], EDGES[-1]); ax.set_ylim(0.5, 50)
    log_axes(ax)
    ax.set_xlabel(xlabel)
    ax.set_ylabel(r"$\sigma_{colour}$/%")
    grid(ax)
axs[0].legend(handles=[Line2D([], [], ls="", marker="s", color="k", ms=3, label=r"$\sqrt{g_X/N_X}$"),
                       Line2D([], [], ls="", marker="s", mfc="none", color="k", ms=3, label=r"$\sqrt{(1-a_X)/N_X}$")],
              fontsize=5, loc="lower right")
axs[3].legend(handles=[Line2D([], [], color=bd.LASER_COLOURS[l], lw=1, label=f"{l} nm") for l in bd.LASERS]
              + [Line2D([], [], ls="", marker=PIXEL_MARKERS[c], color="k", ms=3, label=PIXEL_NAMES[c]) for c in bd.CHANNELS]
              + [Line2D([], [], color="k", lw=0.8, ls="--", label="counting noise")],
              fontsize=5, loc="lower left")
plt.savefig(FIG_DIR / "BeadPrecision_CountingModel.svg", dpi=600, format="svg", bbox_inches="tight")
plt.show()

---
## 3 — Figure: precision by excitation laser

Each laser's beads binned on $N$ (σ_xy) or on that pixel type's $N_X/g_X$ (σ_colour), log-spaced bins;
median and 95 % bootstrap interval for bins with ≥ 5 beads. The lasers agree where their intervals
overlap. The table lists the bins where at least two lasers have ≥ 5 beads, and the largest ratio
between the lasers' medians in each.

In [ ]:
binned = {}
for laser in bd.LASERS:
    d = loc[loc["laser"] == laser]
    binned["sigma_xy", laser] = bd.binned_medians(d["pe"], d["sigma_xy"], EDGES, MIN_PER_BIN, N_BOOT)
    for ch in bd.CHANNELS:
        e = colour[(colour["laser"] == laser) & (colour["channel"] == ch)]
        binned[f"colour_{ch}", laser] = bd.binned_medians(e["n_eff"], e["prec_pct"], EDGES, MIN_PER_BIN, N_BOOT)

panels = [("sigma_xy", r"$\sigma_{xy}$/nm", r"N$_{\mathrm{photoelectrons}}$")] + [
    (f"colour_{ch}", rf"{PIXEL_NAMES[ch]} $\sigma_{{colour}}$/%", r"N$_X$/g$_X$") for ch in bd.CHANNELS]
fig, axs = plotter.two_column_plot(nrows=1, ncols=4, height=2.2)
for ax, (key, ylabel, xlabel) in zip(axs, panels):
    for laser in bd.LASERS:
        b = binned[key, laser].dropna()
        ax.errorbar(b["pe"], b["median"], yerr=[b["median"] - b["lo"], b["hi"] - b["median"]],
                    color=bd.LASER_COLOURS[laser], marker="o", ms=3, lw=1, capsize=2, label=f"{laser} nm")
    ax.set_xscale("log")
    ax.set_xlim(EDGES[0] if key != "sigma_xy" else 100, EDGES[-1])
    log_axes(ax, y=False)
    ax.set_ylim(bottom=0)
    ax.set_xlabel(xlabel)
    ax.set_ylabel(ylabel)
    grid(ax)
axs[0].legend(fontsize=6)
plt.savefig(FIG_DIR / "BeadPrecision_ByExcitation.svg", dpi=600, format="svg", bbox_inches="tight")
plt.show()

rows = []
for key, *_ in panels:
    wide = pd.concat({l: binned[key, l].set_index("pe")[["n", "median", "lo", "hi"]] for l in bd.LASERS}, axis=1)
    med = wide.xs("median", axis=1, level=1)
    shared = med.notna().sum(axis=1) >= 2
    rows.append(wide[shared].assign(quantity=key, max_over_min=(med.max(axis=1) / med.min(axis=1))[shared]))
overlap = pd.concat(rows).reset_index().set_index(["quantity", "pe"])
overlap.to_csv(FIG_DIR / "overlap_binned_medians.csv")
print(overlap.groupby(level="quantity")["max_over_min"].agg(["count", "median", "max"]).round(2))
overlap.round(2)